In [1]:
import slangpy as spy
import pathlib
import matplotlib.pyplot as plt
import numpy as np
import jax
from jax import grad
import jax.numpy as jnp

In [2]:
device = spy.create_device(
    include_paths=[
        pathlib.Path(".").absolute()
    ]
)

[INFO] (rhi) layer: CreateDevice: Debug layer is enabled.
[WARN] No supported shader model found, pretending to support sm_6_0.


In [3]:
program = device.load_program("slang/auto-diff.slang", entry_point_names=["computeCov"])
kernel = device.create_compute_kernel(program)
kernel

ComputeKernel(0x600002d22000)

In [4]:
SIZE=8

In [5]:
gaussian_buf = device.create_buffer(
    element_count=SIZE,
    struct_type=kernel.reflection.gaussians,
    usage=spy.BufferUsage.shader_resource
)
gaussian_buf

Buffer(
  device = 0x107e57628,
  size = 128,
  struct_size = 16,
  format = undefined,
  usage = shader_resource,
  memory_type = device_local,
  memory_usage = 128 B,
  label = 
)

In [6]:
gaussian_cursor = spy.BufferCursor(
    kernel.reflection.gaussians.type_layout.element_type_layout,
    gaussian_buf
)
gaussian_cursor

Object(0x132f6bb88)

In [7]:
for i in range(len(gaussian_cursor)):
    gaussian_cursor[i].write({
        "rotation": float(i) / SIZE * np.pi * 2,
        "scale": np.random.rand(2).astype(np.float32)
    })
gaussian_cursor.apply()

In [8]:
for i in range(len(gaussian_cursor)):
    print(gaussian_cursor[i])

{'rotation': 0.0, 'scale': {0.49580905, 0.27958313}} [Gaussian]
{'rotation': 0.7853981852531433, 'scale': {0.4586963, 0.49253526}} [Gaussian]
{'rotation': 1.5707963705062866, 'scale': {0.25263768, 0.8413995}} [Gaussian]
{'rotation': 2.356194496154785, 'scale': {0.24889666, 0.082398266}} [Gaussian]
{'rotation': 3.1415927410125732, 'scale': {0.7970385, 0.60295653}} [Gaussian]
{'rotation': 3.9269907474517822, 'scale': {0.3308242, 0.6229104}} [Gaussian]
{'rotation': 4.71238899230957, 'scale': {0.9536994, 0.93413}} [Gaussian]
{'rotation': 5.497786998748779, 'scale': {0.7619779, 0.6937027}} [Gaussian]


In [9]:
cov_buf = device.create_buffer(
    element_count=SIZE,
    struct_type=kernel.reflection.covariance,
    usage=spy.BufferUsage.unordered_access
)

In [10]:
cov_cursor = spy.BufferCursor(
    kernel.reflection.covariance.type_layout.element_type_layout,
    cov_buf
)
for i in range(len(cov_cursor)):
    print(cov_cursor[i])

{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]


In [11]:
kernel.dispatch(
    thread_count=[SIZE, 1, 1],
    vars={
        "gaussians": gaussian_buf,
        "covariance": cov_buf
    }
)

In [12]:
cov_cursor = spy.BufferCursor(
    kernel.reflection.covariance.type_layout.element_type_layout,
    cov_buf
)
for i in range(len(cov_cursor)):
    print(cov_cursor[i])

{{0.24582662, 0}, {0, 0.07816672}} [matrix<float,2,2>]
{{0.10520115, 0}, {0, 0.12129549}} [matrix<float,2,2>]
{{1.2195103e-16, 0}, {0, 1.3526757e-15}} [matrix<float,2,2>]
{{0.030974772, 0}, {0, 0.003394737}} [matrix<float,2,2>]
{{0.63527036, 0}, {0, 0.3635566}} [matrix<float,2,2>]
{{0.054722335, 0}, {0, 0.1940087}} [matrix<float,2,2>]
{{1.293395e-16, 0}, {0, 1.24086e-16}} [matrix<float,2,2>]
{{0.29030514, 0}, {0, 0.24061166}} [matrix<float,2,2>]


In [13]:
cov_arr = jnp.array(cov_buf.to_numpy().view(np.float32))
cov_arr

Array([2.4582662e-01, 0.0000000e+00, 0.0000000e+00, 7.8166723e-02,
       1.0520115e-01, 0.0000000e+00, 0.0000000e+00, 1.2129549e-01,
       1.2195103e-16, 0.0000000e+00, 0.0000000e+00, 1.3526757e-15,
       3.0974772e-02, 0.0000000e+00, 0.0000000e+00, 3.3947369e-03,
       6.3527036e-01, 0.0000000e+00, 0.0000000e+00, 3.6355659e-01,
       5.4722335e-02, 0.0000000e+00, 0.0000000e+00, 1.9400869e-01,
       1.2933950e-16, 0.0000000e+00, 0.0000000e+00, 1.2408600e-16,
       2.9030514e-01, 0.0000000e+00, 0.0000000e+00, 2.4061166e-01],      dtype=float32)

In [14]:
mean, grad = jax.value_and_grad(jnp.mean)(cov_arr)
mean, grad

(Array(0.07385419, dtype=float32),
 Array([0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125,
        0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125,
        0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125,
        0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125, 0.03125,
        0.03125, 0.03125, 0.03125, 0.03125], dtype=float32))

In [15]:
cov_diff_buf = device.create_buffer(
    element_count=8,
    struct_type=kernel.reflection.covarianceDiff,
    usage=spy.BufferUsage.shader_resource
)
cov_diff_cursor = spy.BufferCursor(
    kernel.reflection.covarianceDiff.type_layout.element_type_layout,
    cov_diff_buf
)

In [16]:
grad = grad.reshape(SIZE, 2, 2).astype(jnp.float32)
grad.shape

(8, 2, 2)

In [17]:
for i in range(len(cov_diff_cursor)):
    print(cov_diff_cursor[i])

{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]
{{0, 0}, {0, 0}} [matrix<float,2,2>]


In [18]:
for i in range(len(cov_diff_cursor)):
    cov_diff_cursor[i].write(grad[i])
cov_diff_cursor.apply()

In [19]:
for i in range(len(cov_diff_cursor)):
    print(cov_diff_cursor[i])

{{0.03125, 0.03125}, {0.03125, 0.03125}} [matrix<float,2,2>]
{{0.03125, 0.03125}, {0.03125, 0.03125}} [matrix<float,2,2>]
{{0.03125, 0.03125}, {0.03125, 0.03125}} [matrix<float,2,2>]
{{0.03125, 0.03125}, {0.03125, 0.03125}} [matrix<float,2,2>]
{{0.03125, 0.03125}, {0.03125, 0.03125}} [matrix<float,2,2>]
{{0.03125, 0.03125}, {0.03125, 0.03125}} [matrix<float,2,2>]
{{0.03125, 0.03125}, {0.03125, 0.03125}} [matrix<float,2,2>]
{{0.03125, 0.03125}, {0.03125, 0.03125}} [matrix<float,2,2>]


In [20]:
gaussian_diff_buf = device.create_buffer(
    element_count=8,
    struct_type=kernel.reflection.gaussianDiffs,
    usage=spy.BufferUsage.unordered_access
)

In [21]:
gaussian_diff_cursor = spy.BufferCursor(
    kernel.reflection.gaussianDiffs.type_layout.element_type_layout,
    gaussian_diff_buf
)
for i in range(len(gaussian_diff_cursor)):
    print(gaussian_diff_cursor[i])

{'rotation': 0.0, 'scale': {0, 0}} [Gaussian]
{'rotation': 0.0, 'scale': {0, 0}} [Gaussian]
{'rotation': 0.0, 'scale': {0, 0}} [Gaussian]
{'rotation': 0.0, 'scale': {0, 0}} [Gaussian]
{'rotation': 0.0, 'scale': {0, 0}} [Gaussian]
{'rotation': 0.0, 'scale': {0, 0}} [Gaussian]
{'rotation': 0.0, 'scale': {0, 0}} [Gaussian]
{'rotation': 0.0, 'scale': {0, 0}} [Gaussian]


In [22]:
program_bwd = device.load_program("slang/auto-diff.slang", entry_point_names=["computeCovDiff"])
kernel_bwd = device.create_compute_kernel(program_bwd)
kernel_bwd

ComputeKernel(0x600002d57ec0)

In [25]:
kernel_bwd.dispatch(
    thread_count=[SIZE, 1, 1],
    vars={
        "gaussians": gaussian_buf,
        "gaussianDiffs": gaussian_diff_buf,
        "covarianceDiff": cov_diff_buf
    }
)

In [26]:
gaussian_diff_cursor = spy.BufferCursor(
    kernel.reflection.gaussianDiffs.type_layout.element_type_layout,
    gaussian_diff_buf
)
for i in range(len(gaussian_diff_cursor)):
    print(gaussian_diff_cursor[i])

{'rotation': 0.0, 'scale': {0.030988066, 0.017473945}} [Gaussian]
{'rotation': -0.0141560398042202, 'scale': {0.014334259, 0.015391726}} [Gaussian]
{'rotation': 2.1084705092988543e-09, 'scale': {3.0169447e-17, 1.0047811e-16}} [Gaussian]
{'rotation': 0.002148094354197383, 'scale': {0.00777802, 0.0025749458}} [Gaussian]
{'rotation': -5.457514440365685e-09, 'scale': {0.049814906, 0.037684783}} [Gaussian]
{'rotation': -0.015545688569545746, 'scale': {0.010338258, 0.019465951}} [Gaussian]
{'rotation': 1.328239074283033e-09, 'scale': {8.47617e-18, 8.302244e-18}} [Gaussian]
{'rotation': 0.033182308077812195, 'scale': {0.023811806, 0.021678204}} [Gaussian]
